### Implement exponential backoff with Python

In [1]:
import time

In [2]:
class TemporaryAPIError(Exception):
    pass

In [3]:
class SimulatedTimeoutError(Exception):
    pass

In [4]:
def unreliable_api():
    raise TemporaryAPIError("503 Service Unavailable")

In [5]:
def slow_api():
    raise SimulatedTimeoutError("Request timed out")

In [6]:
def call_with_retry(operation, max_attempts=3):
    # Your implementation
    temp = 0
    n = 1

    while temp < max_attempts:
        try:
            result = operation()
            return result
        except (TemporaryAPIError, SimulatedTimeoutError):
            if temp < max_attempts - 1:
                print(f"Attempt {temp+1} Failed: API not available. Retrying after {n} seconds")
                time.sleep(n)
            else:
                print(f"Attempt {temp+1} Failed: No retries remaining")
                raise
            n *= 2
        temp += 1


In [7]:
print(call_with_retry(unreliable_api))

Attempt 1 Failed: API not available. Retrying after 1 seconds
Attempt 2 Failed: API not available. Retrying after 2 seconds
Attempt 3 Failed: No retries remaining


TemporaryAPIError: 503 Service Unavailable

In [8]:
print(call_with_retry(slow_api))

Attempt 1 Failed: API not available. Retrying after 1 seconds
Attempt 2 Failed: API not available. Retrying after 2 seconds
Attempt 3 Failed: No retries remaining


SimulatedTimeoutError: Request timed out

In [9]:
class SimulatedRateLimitError(Exception):
    pass


class MockLLM:
    def __init__(self, max_requests=3):
        self.max_requests = max_requests
        self.request_count = 0

    def generate(self):
        self.request_count += 1

        if self.request_count > self.max_requests:
            raise SimulatedRateLimitError(
                "429: Rate limit exceeded"
            )

        return f"Response {self.request_count}"

In [10]:
llm = MockLLM(max_requests=3)

In [11]:
for _ in range(5):
    try:
        print(llm.generate())
    except SimulatedRateLimitError as e:
        print(e)

Response 1
Response 2
Response 3
429: Rate limit exceeded
429: Rate limit exceeded
